# Sparkle V2 — Fase 3.3: modelos finales, export y evaluación

**Entorno:** Google Colab (GPU T4) · TensorFlow 2.20 + Keras 3
**Requiere:** `dataset_fase3_2_alineado.npz` (caché generada por el notebook 3.2b) o, en su
defecto, `features_v2/` + `embeddings_cnn/`.

Este notebook cierra Fase 3 produciendo los **dos modelos entregables** del proyecto, con sus
métricas completas y sus artefactos de despliegue:

| | Modelo A — *Resumen lineal* | Modelo B — *TCN de fusión* |
| :--- | :--- | :--- |
| Qué es | Regresión logística sobre media y desvío temporal de geometría + embeddings | TCN dilatada con attention pooling, cabeza binaria + ordinal |
| Accuracy Test | **0,6374** (la mejor del proyecto) | 0,5910 |
| Rol | Modelo de mejor rendimiento y *baseline* de referencia | Modelo de producto: da el mapa de calor y la salida ordinal |
| Salidas | probabilidad binaria | probabilidad binaria + ordinal (3 niveles) + pesos de atención |

Se entregan los dos deliberadamente. Que un modelo lineal sobre features promediadas supere a
las arquitecturas secuenciales es un resultado en sí mismo: indica que la señal disponible en
DAiSEE para este target es mayormente **estática**, y funciona como control metodológico de que
la complejidad adicional no estaba aportando.

**Qué produce este notebook:**

1. Los dos modelos entrenados (`.keras`) y convertidos (`.tflite`).
2. `preprocesamiento.npz` con medianas y escaladores — indispensable para inferencia.
3. Métricas completas: accuracy, F1 macro, balanced accuracy, **AUC-ROC**, **average precision**,
   matriz de confusión e intervalos bootstrap.
4. **Análisis de umbral**: la curva de compromiso entre falsos positivos y detección, que es la
   decisión de producto que queda pendiente.
5. Benchmark de latencia de los `.tflite`.
6. `metricas_modelos.json` con todo lo anterior, para citar en la tesis sin volver a correr nada.

In [ ]:
# =====================================================================
# SETUP
# =====================================================================
import sys, os, glob, json, time, shutil, tempfile

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')
assert os.path.isdir(PROJECT_DIR), f'No existe {PROJECT_DIR}'

FEATURES_DIR = f'{PROJECT_DIR}/features_v2'
EMB_DIR      = f'{PROJECT_DIR}/embeddings_cnn'
CACHE_NPZ    = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
OUT_DIR      = f'{PROJECT_DIR}/modelo_final'
os.makedirs(OUT_DIR, exist_ok=True)

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix, classification_report)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print(f'TF {tf.__version__} | Keras {keras.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

SEED = 42
N_FRAMES, N_EMB_FRAMES, EMB_DIM = 60, 16, 128
np.random.seed(SEED); tf.random.set_seed(SEED)

# Config del tronco TCN (la ganadora del diagnostico de Fase 3.1)
CFG = dict(lr=5e-4, ord_weight=0.5, min_epochs=10, lr_patience=6, lr_factor=0.5,
           lr_min=1e-5, label_smoothing=0.1, dropout=0.3, filters=64)
SEEDS = [0, 1, 2, 3, 4]
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 120, 15, 128

## 1. Datos — mismo pipeline que Fase 3.2b

Se reutiliza exactamente el preprocesamiento validado en 3.2b: intersección de clips por
`ClipID`, limpieza de NaN por interpolación, deltas curados sobre 10 features, escalado
ajustado **solo** con el split de entrenamiento y re-split agrupado por participante con el
Test oficial intacto.

Los estadísticos de preprocesamiento (medianas y escaladores) se persisten al final: sin ellos
el modelo no se puede usar en inferencia.

In [ ]:
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen',
                  'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']

def cargar_alineado(split):
    '''Fallback: lee los .npz individuales si no existe la cache de 3.2b.
    Es lento sobre Drive (miles de archivos chicos), pero evita tener que
    volver a correr 3.2b entero solo para generar la cache.'''
    from tqdm.auto import tqdm
    geom = {os.path.splitext(os.path.basename(f))[0]: f
            for f in glob.glob(f'{FEATURES_DIR}/{split}/*.npz')}
    embs = {os.path.splitext(os.path.basename(f))[0]: f
            for f in glob.glob(f'{EMB_DIR}/{split}/*.npz')}
    comunes = sorted(set(geom) & set(embs))
    print(f'{split}: {len(geom)} geometricas, {len(embs)} embeddings, {len(comunes)} en comun')
    Xg, Xe, y3, ids, nombres = [], [], [], [], None
    for cid in tqdm(comunes, desc=f'  {split}', unit='clip'):
        dg = np.load(geom[cid], allow_pickle=True); de = np.load(embs[cid], allow_pickle=True)
        s = dg['sequence'].astype(np.float32); e = de['embeddings'].astype(np.float32)
        if s.shape != (N_FRAMES, 22) or e.shape != (N_EMB_FRAMES, EMB_DIM):
            continue
        if nombres is None:
            nombres = list(dg['feature_names'])
        b = int(dg['boredom'])
        Xg.append(s); Xe.append(e); ids.append(cid)
        y3.append(0 if b == 0 else (1 if b == 1 else 2))
    return (np.stack(Xg), np.stack(Xe), np.array(y3, dtype=np.int64),
            np.array(ids), nombres)

if os.path.exists(CACHE_NPZ):
    d = np.load(CACHE_NPZ, allow_pickle=True)
    Xg_tr, Xe_tr, y3_tr, id_tr = d['Xg_tr'], d['Xe_tr'], d['y3_tr'], d['id_tr']
    Xg_va, Xe_va, y3_va, id_va = d['Xg_va'], d['Xe_va'], d['y3_va'], d['id_va']
    Xg_te, Xe_te, y3_te, id_te = d['Xg_te'], d['Xe_te'], d['y3_te'], d['id_te']
    FEATS22 = list(d['feature_names'])
    print('Cargado de la cache de 3.2b')
else:
    print(f'Sin cache en {CACHE_NPZ} -> leyendo los .npz individuales (lento)')
    Xg_tr, Xe_tr, y3_tr, id_tr, FEATS22 = cargar_alineado('Train')
    Xg_va, Xe_va, y3_va, id_va, _ = cargar_alineado('Validation')
    Xg_te, Xe_te, y3_te, id_te, _ = cargar_alineado('Test')
    os.makedirs(os.path.dirname(CACHE_NPZ), exist_ok=True)
    np.savez(CACHE_NPZ,
             Xg_tr=Xg_tr, Xe_tr=Xe_tr, y3_tr=y3_tr, id_tr=id_tr,
             Xg_va=Xg_va, Xe_va=Xe_va, y3_va=y3_va, id_va=id_va,
             Xg_te=Xg_te, Xe_te=Xe_te, y3_te=y3_te, id_te=id_te,
             feature_names=np.array(FEATS22, dtype=object))
    print('Cache guardada para las proximas corridas')

print(f'Train {Xg_tr.shape} | Validation {Xg_va.shape} | Test {Xg_te.shape}')

# --- Pool Train+Validation, Test oficial aparte ---
Xg_pool = np.concatenate([Xg_tr, Xg_va]); Xe_pool = np.concatenate([Xe_tr, Xe_va])
y3_pool = np.concatenate([y3_tr, y3_va]); id_pool = np.concatenate([id_tr, id_va])
participant_pool = np.array([cid[:6] for cid in id_pool])
y_bin_pool = (y3_pool > 0).astype(np.int64)

def interpolate_sequence(seq):
    df = pd.DataFrame(seq).interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean(X, medians=None, thresh=0.30):
    keep, out = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > thresh:
            continue
        s = interpolate_sequence(X[i])
        if np.isnan(s).any() and medians is not None:
            idx = np.where(np.isnan(s)); s[idx] = np.take(medians, idx[1])
        out.append(s); keep.append(i)
    return np.stack(out), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X, idx=None):
    idx = delta_idx if idx is None else idx
    d_ = np.zeros_like(X[:, :, idx]); d_[:, 1:, :] = np.diff(X[:, :, idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

FEATS32 = FEATS22 + [f'd_{f}' for f in DELTA_FEATURES]

# --- Split interno por participante (para early stopping), Test intacto ---
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx0, va_idx0 = next(sgkf.split(Xg_pool, y_bin_pool, groups=participant_pool))
assert not (set(participant_pool[tr_idx0]) & set(participant_pool[va_idx0])), 'LEAKAGE'

geom_medians = np.nanmedian(Xg_pool[tr_idx0].reshape(-1, 22), axis=0)
emb_medians  = np.nanmedian(Xe_pool[tr_idx0].reshape(-1, EMB_DIM), axis=0)

Xg_a, keep_ga = clean(Xg_pool[tr_idx0], geom_medians)
Xg_b, keep_gb = clean(Xg_pool[va_idx0], geom_medians)
Xe_a, keep_ea = clean(Xe_pool[tr_idx0], emb_medians, thresh=0.5)
Xe_b, keep_eb = clean(Xe_pool[va_idx0], emb_medians, thresh=0.5)

geom_scaler = StandardScaler().fit(add_deltas(Xg_a).reshape(-1, 32))
emb_scaler  = StandardScaler().fit(Xe_a.reshape(-1, EMB_DIM))

def esc_g(X):
    N, T, F = X.shape
    return geom_scaler.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)
def esc_e(X):
    N, T, F = X.shape
    return emb_scaler.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)

def reindex(X, keep_full, keep_final):
    pos = {k: i for i, k in enumerate(keep_full)}
    return X[[pos[k] for k in keep_final]]

keep_a = np.intersect1d(keep_ga, keep_ea); keep_b = np.intersect1d(keep_gb, keep_eb)
Xg_TR = reindex(esc_g(add_deltas(Xg_a)), keep_ga, keep_a)
Xe_TR = reindex(esc_e(Xe_a), keep_ea, keep_a)
Xg_VA = reindex(esc_g(add_deltas(Xg_b)), keep_gb, keep_b)
Xe_VA = reindex(esc_e(Xe_b), keep_eb, keep_b)
y_TR = y_bin_pool[tr_idx0][keep_a]; y3_TR = y3_pool[tr_idx0][keep_a]
y_VA = y_bin_pool[va_idx0][keep_b]

# --- Test oficial ---
Xg_c, keep_gc = clean(Xg_te, geom_medians)
Xe_c, keep_ec = clean(Xe_te, emb_medians, thresh=0.5)
keep_c = np.intersect1d(keep_gc, keep_ec)
Xg_TE = reindex(esc_g(add_deltas(Xg_c)), keep_gc, keep_c)
Xe_TE = reindex(esc_e(Xe_c), keep_ec, keep_c)
y_TE = (y3_te > 0).astype(np.int64)[keep_c]

trivial = max(Counter(y_TE.tolist()).values()) / len(y_TE)
print(f'\nTrain {len(y_TR)} | Val {len(y_VA)} | Test {len(y_TE)} | trivial={trivial:.4f}')

## 2. Modelo A — *Resumen lineal* (el de mejor rendimiento)

Resume cada secuencia por su **media y desvío estándar temporal** y aplica una regresión
logística sobre la concatenación de ambas ramas: `(32 geom × 2) + (128 emb × 2) = 320` valores.

Se ajusta con scikit-learn y después se **transplanta a Keras** como un grafo de una sola capa
densa, para poder exportarlo a TFLite y desplegarlo con el mismo runtime que el Modelo B. La
celda verifica que ambas implementaciones den predicciones idénticas.

In [ ]:
def resumir(X):
    X = np.nan_to_num(X)
    return np.hstack([X.mean(axis=1), X.std(axis=1)])

R_TR = np.hstack([resumir(Xg_TR), resumir(Xe_TR)])
R_VA = np.hstack([resumir(Xg_VA), resumir(Xe_VA)])
R_TE = np.hstack([resumir(Xg_TE), resumir(Xe_TE)])
print(f'Vector de resumen: {R_TR.shape[1]} dimensiones '
      f'(geom mean+std = {Xg_TR.shape[-1]*2}, emb mean+std = {Xe_TR.shape[-1]*2})')

resumen_scaler = StandardScaler().fit(R_TR)
logreg = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
logreg.fit(resumen_scaler.transform(R_TR), y_TR)

p_sk = logreg.predict_proba(resumen_scaler.transform(R_TE))[:, 1]
print(f'sklearn -> acc={accuracy_score(y_TE, (p_sk>0.5).astype(int)):.4f} '
      f'AUC={roc_auc_score(y_TE, p_sk):.4f}')

# ---------------------------------------------------- transplante a Keras
@keras.saving.register_keras_serializable(package='sparkle')
class ResumenTemporal(layers.Layer):
    '''Media y desvio estandar sobre el eje temporal -> (batch, 2*F).'''
    def call(self, x):
        return tf.concat([tf.reduce_mean(x, axis=1), tf.math.reduce_std(x, axis=1)], axis=-1)

def build_modelo_A(batch_size=None):
    inp_g = keras.Input((N_FRAMES, 32), batch_size=batch_size, name='geometricas')
    inp_e = keras.Input((N_EMB_FRAMES, EMB_DIM), batch_size=batch_size, name='embeddings_cnn')
    r = layers.Concatenate(name='resumen')([ResumenTemporal(name='res_geom')(inp_g),
                                            ResumenTemporal(name='res_emb')(inp_e)])
    z = layers.Normalization(axis=-1, name='norm',
                             mean=resumen_scaler.mean_,
                             variance=resumen_scaler.var_)(r)
    out = layers.Dense(1, activation='sigmoid', name='binary')(z)
    m = keras.Model([inp_g, inp_e], out, name='modelo_A_resumen_lineal')
    m.get_layer('binary').set_weights([logreg.coef_.T.astype('float32'),
                                       logreg.intercept_.astype('float32')])
    return m

modelo_A = build_modelo_A()
p_keras = modelo_A.predict([Xg_TE, Xe_TE], verbose=0, batch_size=256).ravel()
print(f'Keras   -> acc={accuracy_score(y_TE, (p_keras>0.5).astype(int)):.4f} '
      f'AUC={roc_auc_score(y_TE, p_keras):.4f}')
print(f'Maxima diferencia sklearn vs Keras: {np.abs(p_sk - p_keras).max():.2e}')
assert np.abs(p_sk - p_keras).max() < 1e-4, 'El transplante a Keras no reproduce sklearn'
print(f'{modelo_A.count_params():,} parametros')
modelo_A.save(f'{OUT_DIR}/modelo_A_resumen_lineal.keras')

## 3. Modelo B — TCN de fusión (el de producto)

Misma arquitectura de Fase 3.2b: dos ramas (embeddings interpolados a 60 pasos + geométricas
normalizadas) que se concatenan por frame, cuatro bloques residuales de `Conv1D` causal dilatada,
attention pooling y dos cabezas — binaria y ordinal aproximada.

**Una corrección respecto de 3.2b:** la rama de embeddings usaba `UpSampling1D(3)` seguido de
`ZeroPadding1D`, lo que dejaba los últimos 12 de 60 pasos en cero y desalineaba temporalmente
las dos ramas. Acá se interpola linealmente de 16 a 60 pasos con `keras.ops.image.resize`,
sin zonas muertas.

In [ ]:
@keras.saving.register_keras_serializable(package='sparkle')
class AttentionPooling(layers.Layer):
    def __init__(self, **kw):
        super().__init__(**kw); self.score_dense = layers.Dense(1, name='attn_score')
    def call(self, x):
        w = tf.nn.softmax(self.score_dense(x), axis=1)
        return tf.reduce_sum(x * w, axis=1), tf.squeeze(w, axis=-1)

@keras.saving.register_keras_serializable(package='sparkle')
class CoralHead(layers.Layer):
    def __init__(self, num_classes, **kw):
        super().__init__(**kw); self.num_classes = num_classes
        self.dense = layers.Dense(num_classes - 1, name='coral_thresholds')
    def call(self, x):
        return tf.nn.sigmoid(self.dense(x))
    def compute_output_shape(self, s): return (s[0], self.num_classes - 1)
    def get_config(self):
        c = super().get_config(); c.update({'num_classes': self.num_classes}); return c

@keras.saving.register_keras_serializable(package='sparkle')
class InterpolarTemporal(layers.Layer):
    '''Interpola linealmente de n_in a n_out pasos (reemplaza UpSampling+ZeroPadding,
    que dejaba 12 de 60 pasos en cero).'''
    def __init__(self, n_out, **kw):
        super().__init__(**kw); self.n_out = n_out
    def call(self, x):
        y = tf.image.resize(tf.expand_dims(x, axis=2), [self.n_out, 1], method='bilinear')
        return tf.squeeze(y, axis=2)
    def compute_output_shape(self, s): return (s[0], self.n_out, s[2])
    def get_config(self):
        c = super().get_config(); c.update({'n_out': self.n_out}); return c

def tcn_block(x, filters, k, dil, dropout, l2, prefix):
    reg = keras.regularizers.l2(l2) if l2 else None
    prev = x
    h = layers.Conv1D(filters, k, padding='causal', dilation_rate=dil,
                      kernel_regularizer=reg, name=f'{prefix}_conv1')(x)
    h = layers.BatchNormalization(name=f'{prefix}_bn1')(h)
    h = layers.Activation('relu', name=f'{prefix}_act1')(h)
    h = layers.Dropout(dropout, name=f'{prefix}_do1')(h)
    h = layers.Conv1D(filters, k, padding='causal', dilation_rate=dil,
                      kernel_regularizer=reg, name=f'{prefix}_conv2')(h)
    h = layers.BatchNormalization(name=f'{prefix}_bn2')(h)
    h = layers.Activation('relu', name=f'{prefix}_act2')(h)
    h = layers.Dropout(dropout, name=f'{prefix}_do2')(h)
    if prev.shape[-1] != filters:
        prev = layers.Conv1D(filters, 1, padding='same', name=f'{prefix}_proj')(prev)
    return layers.Activation('relu', name=f'{prefix}_out')(
        layers.Add(name=f'{prefix}_add')([prev, h]))

def ordinal_loss(y3, p, K=3):
    y3 = tf.cast(y3, tf.float32)
    return tf.reduce_mean(tf.add_n([keras.losses.binary_crossentropy(
        tf.cast(y3 > k, tf.float32), p[:, k]) for k in range(K - 1)]) / (K - 1))

def build_modelo_B(filters=64, dropout=0.3, l2=1e-4, batch_size=None, export=False):
    if export:
        dropout, l2 = 0.0, 0.0
    inp_e = keras.Input((N_EMB_FRAMES, EMB_DIM), batch_size=batch_size, name='embeddings_cnn')
    inp_g = keras.Input((N_FRAMES, 32), batch_size=batch_size, name='geometricas')
    e = InterpolarTemporal(N_FRAMES, name='emb_interp')(inp_e)
    g = layers.BatchNormalization(name='geom_bn')(inp_g)
    x = layers.Concatenate(axis=-1, name='concat')([e, g])
    for i, dil in enumerate((1, 2, 4, 8)):
        x = tcn_block(x, filters, 3, dil, dropout, l2, f'tcn{i}')
    pooled, attn = AttentionPooling(name='attention_pooling')(x)
    h = layers.Dense(32, activation='relu', name='head_dense',
                     kernel_regularizer=keras.regularizers.l2(l2) if l2 else None)(pooled)
    if not export:
        h = layers.Dropout(dropout, name='head_dropout')(h)
    out_bin = layers.Dense(1, activation='sigmoid', name='binary')(h)
    out_ord = CoralHead(3, name='coral')(h)
    outs = [out_bin, out_ord, attn] if export else [out_bin, out_ord]
    return keras.Model([inp_e, inp_g], outs,
                       name='modelo_B_tcn_fusion' + ('_export' if export else ''))

print(f'{build_modelo_B().count_params():,} parametros')

In [ ]:
def smoothed_class_weights(y, beta=0.5):
    c = Counter(y.tolist()); t = len(y)
    w = {int(k): (t / c[k]) ** beta for k in c}
    m = np.mean(list(w.values()))
    return {k: v / m for k, v in w.items()}

def entrenar_B(seed, cfg=CFG, epochs=MAX_EPOCHS, patience=PATIENCE, verbose=False):
    try: keras.utils.set_random_seed(seed)
    except AttributeError: pass
    tf.random.set_seed(seed); np.random.seed(seed)

    model = build_modelo_B(filters=cfg['filters'], dropout=cfg['dropout'])
    opt = keras.optimizers.Adam(learning_rate=cfg['lr'], clipnorm=1.0)
    cw = smoothed_class_weights(y_TR)
    sw = np.array([cw[int(v)] for v in y_TR], dtype='float32')
    ds = tf.data.Dataset.from_tensor_slices(
        ((Xe_TR, Xg_TR), y_TR.astype('float32'), y3_TR.astype('int64'), sw))
    ds = ds.shuffle(4096, seed=seed).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    ord_w, ls = cfg['ord_weight'], cfg['label_smoothing']

    @tf.function
    def step(Xb, yb, y3b, swb):
        with tf.GradientTape() as tape:
            pb, pc = model(list(Xb), training=True)
            l = tf.reduce_mean(swb * keras.losses.binary_crossentropy(
                yb[:, None], pb, label_smoothing=ls)) + ord_w * ordinal_loss(y3b, pc)
            if model.losses: l += tf.add_n(model.losses)
        g = tape.gradient(l, model.trainable_variables)
        g, _ = tf.clip_by_global_norm(g, 1.0)
        opt.apply_gradients(zip(g, model.trainable_variables))

    best, wait, lrw, bw, bep = -1.0, 0, 0, None, -1
    for ep in range(epochs):
        for b in ds: step(*b)
        pv = (model.predict([Xe_VA, Xg_VA], verbose=0, batch_size=256)[0].ravel() > 0.5).astype(int)
        f1 = f1_score(y_VA, pv, average='macro')
        if ep < cfg['min_epochs']: continue
        if bw is None or f1 > best + 1e-4:
            best, wait, lrw, bw, bep = f1, 0, 0, model.get_weights(), ep
        else:
            wait += 1; lrw += 1
            if lrw >= cfg['lr_patience']:
                opt.learning_rate.assign(max(float(opt.learning_rate) * cfg['lr_factor'], cfg['lr_min']))
                lrw = 0
        if wait >= patience: break
    model.set_weights(bw)
    return model, best, bep

resultados_B, mejor_B, mejor_acc = [], None, -1
for s in SEEDS:
    m, vf1, bep = entrenar_B(s)
    p = m.predict([Xe_TE, Xg_TE], verbose=0, batch_size=256)[0].ravel()
    acc = accuracy_score(y_TE, (p > 0.5).astype(int))
    resultados_B.append({'seed': s, 'acc': acc, 'auc': roc_auc_score(y_TE, p),
                         'f1_macro': f1_score(y_TE, (p > 0.5).astype(int), average='macro'),
                         'mejor_epoca': bep})
    print(f'  seed={s}: acc={acc:.4f} auc={roc_auc_score(y_TE, p):.4f} (mejor epoca {bep})')
    if acc > mejor_acc:
        mejor_acc, mejor_B = acc, m

df_B = pd.DataFrame(resultados_B)
print(f"\nModelo B: acc={df_B['acc'].mean():.4f} +/- {df_B['acc'].std():.4f} | "
      f"AUC={df_B['auc'].mean():.4f}")
mejor_B.save(f'{OUT_DIR}/modelo_B_tcn_fusion.keras')
p_B = mejor_B.predict([Xe_TE, Xg_TE], verbose=0, batch_size=256)[0].ravel()

## 4. Evaluación completa — con AUC y average precision

Hasta acá las fases anteriores reportaron accuracy, F1 macro y balanced accuracy, todas
dependientes del umbral fijo de 0,5. **AUC-ROC y average precision son independientes del
umbral** y separan dos preguntas distintas: cuánta capacidad discriminativa tiene el modelo, y
cuánto cuesta operar en un punto de decisión concreto. Un jurado técnico va a esperar ambas.

In [ ]:
def evaluar(nombre, p, y=y_TE, umbral=0.5, n_boot=5000):
    pred = (p > umbral).astype(int)
    m = {'modelo': nombre, 'umbral': umbral,
         'accuracy': accuracy_score(y, pred),
         'f1_macro': f1_score(y, pred, average='macro'),
         'balanced_acc': balanced_accuracy_score(y, pred),
         'auc_roc': roc_auc_score(y, p),
         'average_precision': average_precision_score(y, p),
         'brecha_vs_trivial': accuracy_score(y, pred) - trivial}
    rng = np.random.default_rng(SEED)
    boot = [accuracy_score(y[i], pred[i]) for i in
            (rng.integers(0, len(y), len(y)) for _ in range(n_boot))]
    m['acc_ic95'] = [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))]
    cm = confusion_matrix(y, pred, normalize='true')
    m['recall_atento'], m['recall_desatencion'] = float(cm[0, 0]), float(cm[1, 1])
    return m

met_A = evaluar('A_resumen_lineal', p_keras)
met_B = evaluar('B_tcn_fusion', p_B)
df_met = pd.DataFrame([met_A, met_B]).set_index('modelo')
print(df_met[['accuracy', 'f1_macro', 'balanced_acc', 'auc_roc',
              'average_precision', 'brecha_vs_trivial']].round(4).to_string())
print(f'\nTrivial: {trivial:.4f}')
print(f"\nModelo A - IC95 accuracy: {met_A['acc_ic95']}")
print(f"Modelo B - IC95 accuracy: {met_B['acc_ic95']}")
print('\n--- Modelo A ---')
print(classification_report(y_TE, (p_keras > 0.5).astype(int),
                            target_names=['Atento', 'Desatencion'], digits=3))

## 5. Análisis de umbral — la decisión de producto

Con el umbral por defecto de 0,5 el recall de la clase *Atento* ronda 0,55: casi la mitad de los
alumnos atentos quedarían marcados como desatentos. Para un sistema que alerta al docente esa
tasa de falsos positivos es inusable — las alertas se ignoran en dos clases.

El umbral **no es un hiperparámetro del modelo sino una decisión de producto**, y se puede mover
sin reentrenar nada. Esta celda produce la curva de compromiso para elegirlo con criterio.

In [ ]:
def curva_umbral(p, y=y_TE, nombre=''):
    filas = []
    for u in np.arange(0.30, 0.86, 0.05):
        pred = (p > u).astype(int)
        cm = confusion_matrix(y, pred, normalize='true')
        filas.append({'umbral': round(u, 2),
                      'recall_atento': cm[0, 0], 'recall_desatencion': cm[1, 1],
                      'falsos_pos_atento': cm[0, 1],
                      'accuracy': accuracy_score(y, pred),
                      'f1_macro': f1_score(y, pred, average='macro'),
                      'balanced_acc': balanced_accuracy_score(y, pred)})
    df = pd.DataFrame(filas)
    print(f'\n=== {nombre} ===')
    print(df.round(3).to_string(index=False))
    return df

df_um_A = curva_umbral(p_keras, nombre='Modelo A - resumen lineal')
df_um_B = curva_umbral(p_B, nombre='Modelo B - TCN de fusion')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for nombre, p in [('A - lineal', p_keras), ('B - TCN', p_B)]:
    fpr, tpr, _ = roc_curve(y_TE, p)
    axes[0].plot(fpr, tpr, label=f'{nombre} (AUC={roc_auc_score(y_TE, p):.3f})')
    pr, rc, _ = precision_recall_curve(y_TE, p)
    axes[1].plot(rc, pr, label=f'{nombre} (AP={average_precision_score(y_TE, p):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8); axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR'); axes[0].set_title('Curva ROC'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall (clase Desatencion)'); axes[1].legend()
axes[2].plot(df_um_A['umbral'], df_um_A['recall_atento'], 'o-', label='Recall Atento')
axes[2].plot(df_um_A['umbral'], df_um_A['recall_desatencion'], 's-', label='Recall Desatencion')
axes[2].axvline(0.5, color='k', ls='--', lw=0.8)
axes[2].set_xlabel('Umbral'); axes[2].set_ylabel('Recall')
axes[2].set_title('Compromiso por umbral (Modelo A)'); axes[2].legend()
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/curvas_evaluacion.png', dpi=130); plt.show()

## 6. Export a TFLite y persistencia del preprocesamiento

`preprocesamiento.npz` guarda medianas y escaladores. **Sin ese archivo los modelos son
inservibles en producción**: una entrada sin normalizar con los mismos estadísticos del
entrenamiento produce predicciones arbitrarias.

In [ ]:
def a_tflite(model, specs, rep_dataset=None):
    sm = tempfile.mkdtemp(prefix='sm_'); shutil.rmtree(sm, ignore_errors=True)
    from keras.export import ExportArchive
    ar = ExportArchive(); ar.track(model)
    fn = (lambda x: model(x, training=False)) if len(specs) == 1 \
         else (lambda *xs: model(list(xs), training=False))
    ar.add_endpoint(name='serve', fn=fn, input_signature=list(specs))
    ar.write_out(sm)
    conv = tf.lite.TFLiteConverter.from_saved_model(sm)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    if rep_dataset is not None:
        conv.representative_dataset = rep_dataset
    out = conv.convert(); shutil.rmtree(sm, ignore_errors=True)
    return out

def copiar_pesos(src, dst):
    sm = {l.name: l for l in src.layers}; falt = []
    for l in dst.layers:
        if not l.weights: continue
        s = sm.get(l.name)
        if s is None or not s.weights: falt.append(l.name); continue
        try: l.set_weights(s.get_weights())
        except ValueError: falt.append(f'{l.name}(shape)')
    assert not falt, f'Capas sin pesos copiados: {falt}'
    print(f'  pesos copiados OK ({len([l for l in dst.layers if l.weights])} capas)')

rng = np.random.default_rng(SEED)
i_rep = rng.choice(len(Xg_TR), size=min(200, len(Xg_TR)), replace=False)
rep_g, rep_e = Xg_TR[i_rep], Xe_TR[i_rep]
tam = {}

# ---- Modelo A (batch fijo 1) ----
A_exp = build_modelo_A(batch_size=1)
specs_A = [tf.TensorSpec((1, N_FRAMES, 32), tf.float32, name='geometricas'),
           tf.TensorSpec((1, N_EMB_FRAMES, EMB_DIM), tf.float32, name='embeddings_cnn')]
tfl_A_f = a_tflite(A_exp, specs_A)
_it = tf.lite.Interpreter(model_content=tfl_A_f); _it.allocate_tensors()
orden_A = [tuple(int(v) for v in d['shape'][1:]) for d in _it.get_input_details()]
por_shape = {(N_FRAMES, 32): rep_g, (N_EMB_FRAMES, EMB_DIM): rep_e}
def rep_A():
    for i in range(len(i_rep)):
        yield [por_shape[s][i:i+1].astype('float32') for s in orden_A]
tfl_A = a_tflite(A_exp, specs_A, rep_A)
open(f'{OUT_DIR}/modelo_A_resumen_lineal.tflite', 'wb').write(tfl_A)
tam['modelo_A_int8_kb'] = len(tfl_A) / 1024
print(f'Modelo A TFLite: float {len(tfl_A_f)/1024:.1f} kB | int8 {len(tfl_A)/1024:.1f} kB')

# ---- Modelo B (batch fijo 1, con pesos de atencion) ----
B_exp = build_modelo_B(filters=CFG['filters'], batch_size=1, export=True)
copiar_pesos(mejor_B, B_exp)
specs_B = [tf.TensorSpec((1, N_EMB_FRAMES, EMB_DIM), tf.float32, name='embeddings_cnn'),
           tf.TensorSpec((1, N_FRAMES, 32), tf.float32, name='geometricas')]
tfl_B_f = a_tflite(B_exp, specs_B)
_it = tf.lite.Interpreter(model_content=tfl_B_f); _it.allocate_tensors()
orden_B = [tuple(int(v) for v in d['shape'][1:]) for d in _it.get_input_details()]
def rep_B():
    for i in range(len(i_rep)):
        yield [por_shape[s][i:i+1].astype('float32') for s in orden_B]
tfl_B = a_tflite(B_exp, specs_B, rep_B)
open(f'{OUT_DIR}/modelo_B_tcn_fusion.tflite', 'wb').write(tfl_B)
tam['modelo_B_int8_kb'] = len(tfl_B) / 1024
print(f'Modelo B TFLite: float {len(tfl_B_f)/1024:.1f} kB | int8 {len(tfl_B)/1024:.1f} kB')

# ---- Preprocesamiento (IMPRESCINDIBLE para inferencia) ----
np.savez(f'{OUT_DIR}/preprocesamiento.npz',
         geom_medians=geom_medians, emb_medians=emb_medians,
         geom_scaler_mean=geom_scaler.mean_, geom_scaler_scale=geom_scaler.scale_,
         emb_scaler_mean=emb_scaler.mean_, emb_scaler_scale=emb_scaler.scale_,
         feature_names_22=np.array(FEATS22, dtype=object),
         feature_names_32=np.array(FEATS32, dtype=object),
         delta_features=np.array(DELTA_FEATURES, dtype=object),
         n_frames=N_FRAMES, n_emb_frames=N_EMB_FRAMES, emb_dim=EMB_DIM)
print('preprocesamiento.npz guardado')

ext = f'{PROJECT_DIR}/checkpoints_fase3_2/cnn_extractor.tflite'
if os.path.exists(ext):
    shutil.copy(ext, f'{OUT_DIR}/cnn_extractor.tflite')
    tam['extractor_cnn_int8_kb'] = os.path.getsize(ext) / 1024
    print(f'Extractor CNN copiado ({tam["extractor_cnn_int8_kb"]:.1f} kB)')

# ---- Mismo preprocesamiento en JSON, para la extension de Chrome ----
# El .npz solo se puede leer desde Python. La extension corre los modelos con
# LiteRT.js y necesita los MISMOS vectores de medianas y escaladores; si usara
# otros, el modelo recibiria entradas fuera de su distribucion y devolveria
# predicciones arbitrarias, sin ningun error visible.
prep_web = {
    'geom_medians': geom_medians.tolist(),
    'emb_medians': emb_medians.tolist(),
    'geom_scaler_mean': geom_scaler.mean_.tolist(),
    'geom_scaler_scale': geom_scaler.scale_.tolist(),
    'emb_scaler_mean': emb_scaler.mean_.tolist(),
    'emb_scaler_scale': emb_scaler.scale_.tolist(),
    # El ORDEN de esta lista es la parte critica: el cliente JS valida que
    # coincida con el orden en que produce sus features y falla si no.
    'feature_names_22': list(FEATS22),
    'feature_names_32': list(FEATS32),
    'delta_features': list(DELTA_FEATURES),
    'n_frames': int(N_FRAMES), 'n_emb_frames': int(N_EMB_FRAMES),
    'emb_dim': int(EMB_DIM), 'crop_size': 112,
    'umbral_sugerido': 0.5,
    'trivial_test': float(trivial),
}
with open(f'{OUT_DIR}/preprocesamiento.json', 'w') as f:
    json.dump(prep_web, f)
print(f'preprocesamiento.json guardado ({os.path.getsize(f"{OUT_DIR}/preprocesamiento.json")/1024:.0f} kB)')
print('  -> copiar a frontend/public/models/ junto con los .tflite')

## 7. Benchmark de latencia y volcado de métricas

La latencia se mide sobre el intérprete TFLite en la CPU del runtime de Colab. **No es
representativa de un dispositivo móvil**, pero sirve como cota de orden de magnitud y para
comparar los dos modelos entre sí. Una medición válida para la tesis requiere el hardware
objetivo real.

In [ ]:
def latencia(tflite_bytes, entradas_por_shape, n=100):
    it = tf.lite.Interpreter(model_content=tflite_bytes); it.allocate_tensors()
    dets = it.get_input_details()
    def cargar(i):
        for d in dets:
            s = tuple(int(v) for v in d['shape'][1:])
            arr = entradas_por_shape[s][i:i+1]
            if d['dtype'] == np.int8:
                sc, zp = d['quantization']
                arr = np.clip(np.round(arr / sc + zp), -128, 127).astype(np.int8)
            it.set_tensor(d['index'], arr.astype(d['dtype']))
    for i in range(10): cargar(i); it.invoke()          # calentamiento
    t0 = time.perf_counter()
    for i in range(n): cargar(i % len(Xg_TE)); it.invoke()
    ms = (time.perf_counter() - t0) / n * 1000
    return ms

ent = {(N_FRAMES, 32): Xg_TE, (N_EMB_FRAMES, EMB_DIM): Xe_TE}
lat_A = latencia(tfl_A, ent); lat_B = latencia(tfl_B, ent)
print(f'Latencia por clip (CPU de Colab, int8):')
print(f'  Modelo A: {lat_A:.2f} ms')
print(f'  Modelo B: {lat_B:.2f} ms')
print('  (no incluye la CNN de apariencia ni la extraccion de landmarks)')

reporte = {
    'fecha': time.strftime('%Y-%m-%d'),
    'trivial_test': float(trivial),
    'n_test': int(len(y_TE)), 'n_train': int(len(y_TR)), 'n_val': int(len(y_VA)),
    'tf_version': tf.__version__, 'keras_version': keras.__version__,
    'modelo_A': {**{k: (v if not isinstance(v, np.floating) else float(v))
                    for k, v in met_A.items()},
                 'parametros': int(modelo_A.count_params()),
                 'latencia_ms_cpu': float(lat_A)},
    'modelo_B': {**{k: (v if not isinstance(v, np.floating) else float(v))
                    for k, v in met_B.items()},
                 'parametros': int(mejor_B.count_params()),
                 'latencia_ms_cpu': float(lat_B),
                 'acc_5semillas_media': float(df_B['acc'].mean()),
                 'acc_5semillas_std': float(df_B['acc'].std())},
    'tamanos_kb': {k: float(v) for k, v in tam.items()},
    'umbral_modelo_A': df_um_A.to_dict('records'),
    'umbral_modelo_B': df_um_B.to_dict('records'),
}
with open(f'{OUT_DIR}/metricas_modelos.json', 'w') as f:
    json.dump(reporte, f, indent=2, default=float)
df_B.to_csv(f'{OUT_DIR}/modelo_B_5semillas.csv', index=False)

print('\n=== ARTEFACTOS EN', OUT_DIR, '===')
for f in sorted(os.listdir(OUT_DIR)):
    print(f'  {os.path.getsize(f"{OUT_DIR}/{f}")/1024:9.1f} kB  {f}')

## 8. Qué queda en `modelo_final/`

| Archivo | Para qué sirve |
| :--- | :--- |
| `modelo_A_resumen_lineal.keras` / `.tflite` | Modelo de mejor rendimiento |
| `modelo_B_tcn_fusion.keras` / `.tflite` | Modelo de producto (mapa de calor + salida ordinal) |
| `cnn_extractor.tflite` | Rama de apariencia — se ejecuta antes que cualquiera de los dos |
| `preprocesamiento.npz` | Medianas y escaladores. **Imprescindible**: sin esto las predicciones son arbitrarias |
| `metricas_modelos.json` | Todas las métricas y curvas de umbral, para citar en la tesis |
| `curvas_evaluacion.png` | ROC, precision-recall y compromiso por umbral |

El siguiente notebook, `Sparkle_V2_Inferencia_End2End.ipynb`, consume estos artefactos y hace el
recorrido completo desde un video hasta la predicción.